# SQLintel — ML false-positive triage classifier (v1 baseline)

Trains a **free, local** classifier that refines the deterministic engine's confidence.
Run this on your machine or on **Google Colab / Kaggle** (free GPU not even needed for the baseline).

Output: `models/sqli_clf.joblib`, which `sqlintel/ml/triage.py` auto-loads at scan time.

**Pipeline:** labeled payloads/requests → `finding_features()` → XGBoost/RandomForest → export.

> The feature function is imported from the package so training and inference stay identical.
> v2 upgrade (documented at the bottom): fine-tune DistilBERT/CodeBERT on request+response context.

In [ ]:
# If running on Colab, clone your repo first so we can import the SAME feature code:
# !git clone https://github.com/<you>/sqlintel && %cd sqlintel
!pip install -q scikit-learn xgboost pandas joblib

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # import the sqlintel package from repo root
from sqlintel.ml.features import finding_features
import pandas as pd, numpy as np

## 1. Build a labeled dataset

Two sources, both free:
1. **Public payload sets** — e.g. Kaggle 'SQL Injection Dataset' (malicious vs benign strings).
2. **Self-generated** — run the SQLintel engine against DVWA/Juice Shop and log every
   (technique, confidence, payload, was_it_real) tuple. This is your differentiator.

Below is a small **seed** dataset so the notebook runs end-to-end today; replace/extend it
with the real data as you collect it.

In [ ]:
# label 1 = genuine injection signal, 0 = benign / false positive
seed = [
    ('error-based',   0.90, "'", 1),
    ('error-based',   0.90, "' OR '1'='1", 1),
    ('boolean-based', 0.80, "' AND 1=1-- -", 1),
    ('time-based',    0.85, "' AND SLEEP(5)-- -", 1),
    ('time-based',    0.85, "'; WAITFOR DELAY '0:0:5'-- -", 1),
    ('boolean-based', 0.62, "' AND 1=2-- -", 1),
    # benign / reflected values that a naive scanner might over-flag:
    ('boolean-based', 0.61, "O'Brien", 0),
    ('error-based',   0.55, "search term", 0),
    ('boolean-based', 0.60, "12345", 0),
    ('error-based',   0.50, "user@example.com", 0),
    ('time-based',    0.50, "normal input", 0),
    ('boolean-based', 0.58, "blue widget", 0),
]
rows = [ {**finding_features(t, c, p), 'label': y} for (t, c, p, y) in seed ]
df = pd.DataFrame(rows).fillna(0.0)
X = df.drop(columns=['label'])
y = df['label']
print(df.shape); df.head()

## 2. Train the baseline classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# XGBoost if available, else RandomForest — both export cleanly to joblib.
try:
    from xgboost import XGBClassifier
    clf = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                        eval_metric='logloss')
except Exception:
    from sklearn.ensemble import RandomForestClassifier
    clf = RandomForestClassifier(n_estimators=300, random_state=0)

# With a tiny seed set we train on all of it; switch to a real split once you have data.
clf.fit(X, y)
print(classification_report(y, clf.predict(X)))

## 3. Export — SQLintel picks this up automatically

In [ ]:
import joblib
os.makedirs('../models', exist_ok=True)
joblib.dump(clf, '../models/sqli_clf.joblib')
print('Saved models/sqli_clf.joblib — triage.py will now blend model probability into confidence.')

## v2 roadmap — transformer upgrade (the research headline)

1. Collect request+response **context** (not just the payload) into a text field.
2. Fine-tune `distilbert-base-uncased` or `microsoft/codebert-base` for binary
   SQLi classification (HuggingFace `Trainer`, free Colab GPU).
3. Compare precision/recall/FP-rate vs this baseline **and vs sqlmap/Ghauri** on
   DVWA + Juice Shop — that comparison table is your project's headline result.
4. Wrap the transformer behind the same `predict_proba(DataFrame)` interface so
   `triage.py` needs no changes.